**MIPACE Feature Extraction**

In [121]:
import pandas as pd
import os 
import glob

In [124]:
path = r'visit_data'
filenames = glob.glob(path + "/*.csv")

dfs = []
for filename in filenames:
    dfs.append(pd.read_csv(filename))

# Concatenate all data into one DataFrame
big_frame = pd.concat(dfs, ignore_index=True)

#Add stage number
def stage_number(visit_df):
    visit_df['Stage'] = visit_df[visit_df['Phase'] == 'EXERCISE'].groupby(['Speed','Grade']).ngroup() + 1
    return visit_df

big_frame = big_frame.groupby(['participant_ID', 'visit_date']).apply(stage_number, include_groups=False).reset_index()

#Convert t to datetime
big_frame['t'] = pd.to_datetime('1900-01-01 ' + big_frame['t'], format='%Y-%m-%d %H:%M:%S') ## use dummy date for day
big_frame

,participant_ID,visit_date,level_2,Local Time,t,Rf,VT,VE,IV,VO2,...,Phase time,VO2/Kg%Pred,BR,VT/Ti,HRR,PaCO2_e,Dyspnea,SV e,CO e,Stage
0,TR000100,20220203,33726,NaN,1900-01-01 00:00:01,23.72,0.823,19.518,509,832.406802,...,00:00:00,45,83.5,0.50,129.0,39,NaN,178.3,9.6,NaN
1,TR000100,20220203,33727,NaN,1900-01-01 00:00:06,11.45,0.851,9.744,825,425.775504,...,00:00:05,23,91.8,0.33,129.0,39,NaN,109.3,5.9,NaN
2,TR000100,20220203,33728,NaN,1900-01-01 00:00:16,6.22,2.646,16.469,2497,546.955374,...,00:00:15,30,86.1,0.76,119.0,36,NaN,111.8,7.2,NaN
3,TR000100,20220203,33729,NaN,1900-01-01 00:00:20,14.08,0.467,6.577,562,243.244898,...,00:00:19,13,94.4,0.21,119.0,37,NaN,57.8,3.7,NaN
4,TR000100,20220203,33730,NaN,1900-01-01 00:00:23,20.62,0.721,14.866,395,573.933809,...,00:00:22,31,87.4,0.79,121.0,38,NaN,119.6,7.4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47156,TR000110,20220901,32225,NaN,1900-01-01 00:42:45,37.04,0.433,16.037,395,369.765180,...,00:19:55,16,87.0,0.79,94.0,25,LEVEL_20,60.6,5.3,NaN
47157,TR000110,20220901,32226,NaN,1900-01-01 00:42:47,30.61,0.450,13.776,429,316.504152,...,00:19:57,13,88.9,0.63,95.0,23,LEVEL_20,53.9,4.7,NaN
47158,TR000110,20220901,32227,NaN,1900-01-01 00:42:49,33.33,0.324,10.800,291,227.311298,...,00:19:59,10,91.3,0.48,95.0,25,LEVEL_20,40.4,3.5,NaN
47159,TR000110,20220901,32228,NaN,1900-01-01 00:42:50,34.48,0.337,11.621,428,247.015659,...,00:20:00,10,90.6,0.53,95.0,22,LEVEL_20,43.5,3.8,NaN


In [125]:
#Split Up Dataframes
def create_dataframe_dictionary(parent_df):
    participant_registry = dict()
    for row in parent_df.itertuples():
        participant_registry[row.participant_ID] = dict()
    for row in parent_df.itertuples():
        participant_registry[row.participant_ID][row.visit_date] = []
    for row in parent_df.itertuples():
        participant_registry[row.participant_ID][row.visit_date].append(row)
    participant_df = dict()
    for participant_id, participant_data in participant_registry.items():
        for visit_date in participant_data.keys():
            participant_df[f'{participant_id}_{visit_date}'] = pd.DataFrame(participant_registry[participant_id][visit_date])
    print(participant_df.keys())
    return participant_df

participant_df = create_dataframe_dictionary(big_frame)
participant_df['TR000105_20220207']



dict_keys(['TR000100_20220203', 'TR000100_20220331', 'TR000100_20220623', 'TR000101_20220203', 'TR000101_20220331', 'TR000101_20220630', 'TR000102_20220204', 'TR000102_20220401', 'TR000102_20220630', 'TR000103_20220204', 'TR000103_20220401', 'TR000103_20220706', 'TR000104_20220207', 'TR000104_20220314', 'TR000104_20220627', 'TR000105_20220207', 'TR000105_20220517', 'TR000105_20220920', 'TR000106_20220208', 'TR000106_20220420', 'TR000106_20220823', 'TR000107_20220208', 'TR000107_20220519', 'TR000107_20220805', 'TR000108_20220209', 'TR000109_20220209', 'TR000109_20220518', 'TR000109_20220824', 'TR000110_20220210', 'TR000110_20220428', 'TR000110_20220901'])


,Index,participant_ID,visit_date,level_2,_4,t,Rf,VT,VE,IV,...,_87,_88,BR,_90,HRR,PaCO2_e,Dyspnea,_94,_95,Stage
0,21511,TR000105,20220207,0,14:35:51,1900-01-01 00:00:01,34.48,0.495,17.069,278,...,00:00:00,48,79.1,0.81,NaN,35,NaN,NaN,NaN,NaN
1,21512,TR000105,20220207,1,14:35:54,1900-01-01 00:00:04,22.39,0.386,8.642,423,...,00:00:03,18,89.4,0.45,89.0,30,NaN,45.3,3.6,NaN
2,21513,TR000105,20220207,2,14:35:56,1900-01-01 00:00:06,31.58,0.215,6.789,155,...,00:00:05,12,91.7,0.26,89.0,24,NaN,30.3,2.4,NaN
3,21514,TR000105,20220207,3,14:35:57,1900-01-01 00:00:07,45.11,0.361,16.286,434,...,00:00:06,37,80.1,0.55,91.0,30,NaN,80.6,6.2,NaN
4,21515,TR000105,20220207,4,14:35:59,1900-01-01 00:00:09,39.74,0.478,18.993,307,...,00:00:08,46,76.8,0.89,91.0,33,NaN,94.8,7.3,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1484,22995,TR000105,20220207,1484,15:24:44,1900-01-01 00:48:54,7.63,0.565,4.313,784,...,00:19:51,12,94.7,0.35,70.0,35,LEVEL_19,25.6,2.5,NaN
1485,22996,TR000105,20220207,1485,15:24:46,1900-01-01 00:48:56,19.35,0.402,7.781,345,...,00:19:53,18,90.5,0.23,70.0,32,LEVEL_19,35.3,3.5,NaN
1486,22997,TR000105,20220207,1486,15:24:48,1900-01-01 00:48:58,24.90,0.360,8.963,215,...,00:19:55,20,89.0,0.63,70.0,34,LEVEL_19,38.9,3.8,NaN
1487,22998,TR000105,20220207,1487,15:24:49,1900-01-01 00:48:59,41.38,0.944,39.062,370,...,00:19:56,115,52.2,1.57,70.0,38,LEVEL_19,124.5,12.2,NaN


In [126]:
def test_keys(df_dictionary):
    for key, values in df_dictionary.items():
        for row in values.itertuples():
            assert key == f'{row.participant_ID}_{row.visit_date}'
    return True

test_keys(participant_df)

True

In [127]:
#Create VO2 Outcome DF
def create_outcome_df(dictionary_df):
    id_list = list(dictionary_df.keys())
    result_df = pd.DataFrame({
    "participant_visit_id": id_list,
    "VO2_peak": None,
    "VO2/kgPeak": None,
    "True VO2max": None,
    #"VO2Peak start time- global": None,
    #"VO2Peak end time- global": None,
    "VO2Peak start time- local": None,
    "VO2Peak end time- local": None,
    "RQPeak": None,
    "HRPeak": None,
    "GradePeak": None,
    "SpeedPeak": None,
    "MarkerPeak": None,
    "EEMPeak": None,
    "Fat%Peak": None,
    "CHO%Peak": None,
    "Lactate-VO2Peak": None,
    "RPE-VO2peak": None,
    })

    # Set participant_id as the index
    result_df.set_index("participant_visit_id", inplace=True)
    return result_df
create_outcome_df(participant_df)
VO2_outcome_df = create_outcome_df(participant_df)
VO2_outcome_df

,VO2_peak,VO2/kgPeak,True VO2max,VO2Peak start time- local,VO2Peak end time- local,RQPeak,HRPeak,GradePeak,SpeedPeak,MarkerPeak,EEMPeak,Fat%Peak,CHO%Peak,Lactate-VO2Peak,RPE-VO2peak
participant_visit_id,,,,,,,,,,,,,,,
TR000100_20220203,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000100_20220331,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000100_20220623,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000101_20220203,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000101_20220331,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000101_20220630,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000102_20220204,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000102_20220401,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
TR000102_20220630,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [128]:
# Get Visit Number
VO2_outcome_df['ID'] = VO2_outcome_df.index.str.split('_').str[0]
VO2_outcome_df['Visit Date'] = VO2_outcome_df.index.str.split('_').str[1]
VO2_outcome_df = VO2_outcome_df.sort_values(by=['ID', 'Visit Date'])
VO2_outcome_df['Visit Number'] = VO2_outcome_df.groupby('ID').cumcount()+1

In [134]:

def get_VO2_peak_and_time(id, test_df, VO2_outcome_df):
    #id = f'{test_df['participant_ID'][0]}_{test_df['visit_date'][0]}'
    #test_df['t'] = pd.to_datetime('1900-01-01 ' + test_df['t'], format='%Y-%m-%d %H:%M:%S')
    rolling_avg = test_df.rolling('30s', on='t')['VO2'].mean()
    max_avg = rolling_avg.max()
    end_index = rolling_avg.idxmax()
    end_time = test_df.loc[end_index, 't']
    start_index = test_df.loc[test_df['t'] <= end_time - pd.Timedelta(seconds=30)].index[-1]
    start_time = test_df.loc[start_index, 't']
    VO2_outcome_df.loc[id, 'VO2_peak'] = max_avg
    VO2_outcome_df.loc[id, 'VO2Peak start time- local'] = start_time.strftime('%H:%M:%S')
    VO2_outcome_df.loc[id, 'VO2Peak end time- local'] = end_time.strftime('%H:%M:%S')
    return

for id, df in participant_df.items():
    get_VO2_peak_and_time(id, df, VO2_outcome_df)

VO2_outcome_df.head()

,VO2_peak,VO2/kgPeak,True VO2max,VO2Peak start time- local,VO2Peak end time- local,RQPeak,HRPeak,GradePeak,SpeedPeak,MarkerPeak,EEMPeak,Fat%Peak,CHO%Peak,Lactate-VO2Peak,RPE-VO2peak,ID,Visit Date,Visit Number
participant_visit_id,,,,,,,,,,,,,,,,,,
TR000100_20220203,3105.897672,None,None,00:22:25,00:22:55,None,None,None,None,None,None,None,None,None,None,TR000100,20220203,1
TR000100_20220331,3451.548659,None,None,00:20:39,00:21:09,None,None,None,None,None,None,None,None,None,None,TR000100,20220331,2
TR000100_20220623,3453.436232,None,None,00:26:29,00:26:59,None,None,None,None,None,None,None,None,None,None,TR000100,20220623,3
TR000101_20220203,2641.110769,None,None,00:21:48,00:22:19,None,None,None,None,None,None,None,None,None,None,TR000101,20220203,1
TR000101_20220331,2653.500411,None,None,00:23:36,00:24:06,None,None,None,None,None,None,None,None,None,None,TR000101,20220331,2


In [7]:
def get_VO2_kg_peak(df_dictionary, result_df):
    id_list = list(result_df.index)
    for id in id_list:
        df = df_dictionary[id]
        
        # Convert 'timestamp' to seconds elapsed
        df['seconds_elapsed'] = df['t'].apply(
            lambda x: int(x.split(':')[0]) * 3600 + int(x.split(':')[1]) * 60 + int(x.split(':')[2])
        )
        
        # Initialize variables for tracking VO2_peak and VO2/kgPeak
        peak_value = float('-inf')
        peak_start_time = None
        peak_end_time = None
        kg_peak_value = None

        # Iterate through each row to compute the rolling mean based on a 30-second window
        for i, current_row in df.iterrows():
            current_time = current_row['seconds_elapsed']
            window_end = current_time
            window_start = current_time - 30
            
            # Select rows within the 30-second window
            window_data_VO2 = df[(df['seconds_elapsed'] >= window_start) & (df['seconds_elapsed'] <= window_end)]['VO2']
            rolling_mean_VO2 = window_data_VO2.mean()

            # Update the peak value and corresponding time window
            if rolling_mean_VO2 > peak_value:
                peak_value = rolling_mean_VO2
                peak_start_time = window_start
                peak_end_time = window_end

                # Find the rolling mean of the _16 column for the same interval
                window_data_16 = df[(df['seconds_elapsed'] >= window_start) & (df['seconds_elapsed'] <= window_end)]['_16']
                kg_peak_value = window_data_16.mean()

        # Convert seconds back to HH:MM:SS format
        def seconds_to_hhmmss(seconds):
            hours = seconds // 3600
            minutes = (seconds % 3600) // 60
            seconds = seconds % 60
            return f"{hours:02}:{minutes:02}:{seconds:02}"

        # Record results in the result DataFrame
        result_df.loc[id, 'VO2_peak'] = peak_value
        result_df.loc[id, 'VO2Peak start time- global'] = seconds_to_hhmmss(peak_start_time)
        result_df.loc[id, 'VO2Peak end time- global'] = seconds_to_hhmmss(peak_end_time)
        result_df.loc[id, 'VO2/kgPeak'] = kg_peak_value
    return result_df

VO2_outcome_df = get_VO2_kg_peak(participant_df, VO2_outcome_df)
VO2_outcome_df
   

,VO2_peak,VO2/kgPeak,True VO2max,VO2Peak start time- global,VO2Peak end time- global,VO2Peak start time- local,VO2Peak end time- local,RQPeak,HRPeak,GradePeak,SpeedPeak,MarkerPeak,EEMPeak,Fat%Peak,CHO%Peak,Lactate-VO2Peak,RPE-VO2peak,ID,Visit Date,Visit Number
participant_visit_id,,,,,,,,,,,,,,,,,,,,
TR000100_20220203,3103.659316,59.572174,None,00:22:26,00:22:56,None,None,None,None,None,None,None,None,None,None,None,None,TR000100,20220203,1
TR000100_20220331,3443.011822,67.247059,None,00:20:39,00:21:09,None,None,None,None,None,None,None,None,None,None,None,None,TR000100,20220331,2
TR000100_20220623,3447.64189,66.685769,None,00:26:30,00:27:00,None,None,None,None,None,None,None,None,None,None,None,None,TR000100,20220623,3
TR000101_20220203,2651.252137,47.5136,None,00:21:50,00:22:20,None,None,None,None,None,None,None,None,None,None,None,None,TR000101,20220203,1
TR000101_20220331,2647.141505,47.439655,None,00:23:00,00:23:30,None,None,None,None,None,None,None,None,None,None,None,None,TR000101,20220331,2
TR000101_20220630,2573.773731,45.634074,None,00:22:49,00:23:19,None,None,None,None,None,None,None,None,None,None,None,None,TR000101,20220630,3
TR000102_20220204,3547.730784,54.329167,None,00:24:36,00:25:06,None,None,None,None,None,None,None,None,None,None,None,None,TR000102,20220204,1
TR000102_20220401,3803.85079,59.0656,None,00:28:41,00:29:11,None,None,None,None,None,None,None,None,None,None,None,None,TR000102,20220401,2
TR000102_20220630,3789.272083,56.7264,None,00:27:20,00:27:50,None,None,None,None,None,None,None,None,None,None,None,None,TR000102,20220630,3


Index(['Index', 'participant_ID', 'visit_date', '_3', 't', 'Rf', 'VT', 'VE',
       'IV', 'VO2', 'VCO2', 'RQ', 'O2exp', 'CO2exp', '_14', '_15', '_16',
       'METS', 'HR', '_19', 'FeO2', 'FeCO2', 'FetO2', 'FetCO2', 'FiO2',
       'FiCO2', 'PeO2', 'PeCO2', 'PetO2', 'PetCO2', 'SpO2', 'Grade', 'Speed',
       'Phase', 'Marker', '_35', '_36', '_37', 'PB', 'EEkc', 'EEh', 'EEm',
       'EEtot', 'EEkg', 'PRO', 'Fat', 'CHO', '_47', '_48', '_49', 'npRQ',
       '_51', '_52', '_53', '_54', '_55', '_56', '_57', '_58', '_59', '_60',
       '_61', '_62', '_63', '_64', '_65', '_66', '_67', '_68', '_69', '_70',
       '_71', '_72', '_73', '_74', '_75', '_76', 'Ti', 'Te', 'Ttot', '_80',
       '_81', 'LogVE', '_83', '_84', '_85', '_86', '_87', 'BR', '_89', 'HRR',
       'PaCO2_e', 'Dyspnea', '_93', '_94'],
      dtype='object')

In [8]:
#Create Result Dataframes
def create_results_dictionary(parent_df):
    participant_registry = dict()
    for row in parent_df.itertuples():
        participant_registry[row.participant_ID] = dict()
    for row in parent_df.itertuples():
        participant_registry[row.participant_ID][row.visit_date] = []
    participant_result_df = dict()
    columns = ["V02max", "LT1", "LT2", "VT1", "VT2", "Stage 1", "Stage 2", "Stage 3", "... Stage X"]
    rows = [
        "VO2 (mL/min/kg)",
        "Velocity (km/h)",
        "Grade (%)",
        "Stage (#)",
        "RPE (#)",
        "HR (bpm)",
        "Lactate (mmol/L)",
        "RQ",
        "True VO2max? Y/N",
        "Reach Steady State?",
    ]
    for participant_id, participant_data in participant_registry.items():
        for visit_date in participant_data.keys():
            participant_result_df[f'{participant_id}_{visit_date}'] = pd.DataFrame(columns=columns, index=rows)
    print(participant_df.keys())
    return participant_result_df

participant_result_df = create_results_dictionary(big_frame)
participant_result_df['TR000105_20220207']


dict_keys(['TR000105_20220207', 'TR000105_20220517', 'TR000105_20220920', 'TR000106_20220420', 'TR000106_20220208', 'TR000106_20220823', 'TR000102_20220204', 'TR000102_20220401', 'TR000102_20220630', 'TR000100_20220623', 'TR000100_20220331', 'TR000100_20220203', 'TR000104_20220627', 'TR000104_20220207', 'TR000104_20220314', 'TR000109_20220209', 'TR000109_20220824', 'TR000109_20220518', 'TR000107_20220519', 'TR000107_20220208', 'TR000107_20220805', 'TR000103_20220401', 'TR000103_20220204', 'TR000103_20220706', 'TR000101_20220203', 'TR000101_20220630', 'TR000101_20220331', 'TR000110_20220210', 'TR000110_20220428', 'TR000110_20220901', 'TR000108_20220209'])


,V02max,LT1,LT2,VT1,VT2,Stage 1,Stage 2,Stage 3,... Stage X
VO2 (mL/min/kg),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Velocity (km/h),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Grade (%),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Stage (#),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RPE (#),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HR (bpm),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Lactate (mmol/L),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
True VO2max? Y/N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Reach Steady State?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
